# Gráficos Comparación Temporadas y Estaciones

In [ ]:
import json
from pathlib import Path

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd

# Definición de la paleta de colores corporativa y estilos base de Matplotlib.
VIGNO_TERRACOTTA = "#9C4A27"
VIGNO_BLACK = "#1A1A1A"
VIGNO_WHITE = "#FFFFFF"
VIGNO_GRAY_DARK = "#4A4A4A"
VIGNO_GRAY_LIGHT = "#E0E0E0"
VIGNO_ACCENT_GOLD = "#D4AF37"
VIGNO_ACCENT_BLUE = "#2C5F78"

plt.rcParams["font.family"] = "sans-serif"
plt.rcParams["text.color"] = VIGNO_BLACK
plt.rcParams["axes.labelcolor"] = VIGNO_GRAY_DARK
plt.rcParams["xtick.color"] = VIGNO_GRAY_DARK
plt.rcParams["ytick.color"] = VIGNO_GRAY_DARK

# Rutas seguras adaptadas para Jupyter y scripts
RAIZ = Path.cwd() if (Path.cwd() / "manage.py").exists() else Path.cwd().parent
datos_path = RAIZ / "data" / "cliente" / "vigno-datos-reales.json"
output_dir = RAIZ / "data" / "cliente" / "graficos"
output_dir.mkdir(parents=True, exist_ok=True)

with open(datos_path, encoding="utf-8") as f:
    data = json.load(f)

fichas = data.get("ficha_temporada", [])
resumenes = data.get("resumen_diario", [])
temp_curso = data.get("temporada_en_curso", "")

# Diccionarios para arreglar las etiquetas y tildes según el frontend
etiquetas_termica = {"calida": "Cálida", "fria": "Fría", "normal": "Normal"}

etiquetas_hidrica = {"lluviosa": "Lluviosa", "seca": "Seca", "normal": "Normal"}

# Gráfico 1: Panel 2x2 histórico de Winkler y Lluvia Invernal para San Clemente y El Arenal.
estaciones_info = {
    "cp-san-clemente-01": {"nombre": "San Clemente (Precordillera)", "fila": 0},
    "cp-villavicencio-01": {"nombre": "El Arenal (Secano)", "fila": 1},
}

fig, axes = plt.subplots(2, 2, figsize=(16, 14), layout="constrained")

for est_id, info in estaciones_info.items():
    # SE QUITA EL FILTRO GLOBAL DE temp_curso PARA NO AFECTAR A LA LLUVIA INVERNAL
    fichas_est = [f for f in fichas if f["estacion_id"] == est_id]
    fichas_est.sort(key=lambda x: x["temporada"])

    fila = info["fila"]
    ax_w = axes[fila, 0]
    ax_ll = axes[fila, 1]

    # --------- WINKLER ---------
    w_vals = []
    w_temporadas = []
    w_colors = []
    w_labels = []

    for f in fichas_est:
        temp = f["temporada"]

        # Winkler en curso se omite porque el verano no ha terminado
        if temp == temp_curso:
            continue

        w = f["indices"]["winkler"]

        incompleta_w = not w["calidad_dato"]["confiable"]
        termica = (f.get("clasificacion") or {}).get("termica")

        if incompleta_w:
            w_vals.append(0)
            w_temporadas.append(temp)
            w_colors.append("none")
            w_labels.append("Incompleta")
            continue

        v = w.get("valor")
        if v is not None:
            w_vals.append(v)
            w_temporadas.append(temp)

            lbl_mostrar = etiquetas_termica.get(termica, "Sin clasificar")
            w_labels.append(lbl_mostrar)

            if termica == "calida":
                w_colors.append(VIGNO_TERRACOTTA)
            elif termica == "fria":
                w_colors.append(VIGNO_ACCENT_BLUE)
            elif termica == "normal":
                w_colors.append(VIGNO_GRAY_DARK)
            else:
                w_colors.append(VIGNO_GRAY_LIGHT)

    vals_validos_w = [v for v, lbl in zip(w_vals, w_labels, strict=False) if lbl != "Incompleta"]
    promedio_w = sum(vals_validos_w) / len(vals_validos_w) if vals_validos_w else 0

    bars_w = ax_w.bar(w_temporadas, w_vals, color=w_colors, zorder=3)
    ax_w.axhline(
        promedio_w, color=VIGNO_BLACK, linestyle="--", zorder=4, label=f"Promedio: {promedio_w:.1f}"
    )
    ax_w.set_title(
        f"Índice Winkler - {info['nombre']}", fontweight="bold", color=VIGNO_TERRACOTTA, pad=15
    )
    ax_w.set_ylabel("Grados Día")
    ax_w.set_ylim(0, max(vals_validos_w) * 1.25 if vals_validos_w else 2000)
    ax_w.tick_params(axis="x", rotation=45)
    ax_w.grid(axis="y", linestyle=":", alpha=0.6)
    ax_w.spines["top"].set_visible(False)
    ax_w.spines["right"].set_visible(False)
    ax_w.legend(loc="upper left")

    for bar, lbl in zip(bars_w, w_labels, strict=False):
        if lbl == "Incompleta":
            ax_w.text(
                bar.get_x() + bar.get_width() / 2,
                50,
                "Incompleta",
                ha="center",
                va="bottom",
                rotation=90,
                color=VIGNO_BLACK,
                fontweight="bold",
                fontsize=11,
                fontstyle="italic",
            )
        else:
            yval = bar.get_height()
            ax_w.text(
                bar.get_x() + bar.get_width() / 2,
                yval + 15,
                f"{yval:.0f}",
                ha="center",
                va="bottom",
                color=VIGNO_BLACK,
                fontweight="bold",
                fontsize=10,
            )
            ax_w.text(
                bar.get_x() + bar.get_width() / 2,
                100,
                lbl,
                ha="center",
                va="bottom",
                rotation=90,
                color=VIGNO_WHITE if lbl != "Sin clasificar" else VIGNO_BLACK,
                fontweight="bold",
                fontsize=11,
            )

    # --------- LLUVIA INVERNAL ---------
    ll_vals = []
    ll_temporadas = []
    ll_colors = []
    ll_labels = []

    for f in fichas_est:
        temp = f["temporada"]
        # La Lluvia Invernal SÍ incluye la temporada en curso porque el invierno ya terminó.
        # Solo se delega la responsabilidad a calidad_dato.confiable.

        ll = f["lluvia_invernal"]

        incompleta_ll = not ll["calidad_dato"]["confiable"]
        hidrica = (f.get("clasificacion") or {}).get("hidrica")

        if incompleta_ll:
            ll_vals.append(0)
            ll_temporadas.append(temp)
            ll_colors.append("none")
            ll_labels.append("Incompleta")
            continue

        v = ll.get("acumulado_mm")
        if v is not None:
            ll_vals.append(v)
            ll_temporadas.append(temp)

            lbl_mostrar = etiquetas_hidrica.get(hidrica, "Sin clasificar")
            ll_labels.append(lbl_mostrar)

            if hidrica == "lluviosa":
                ll_colors.append(VIGNO_ACCENT_BLUE)
            elif hidrica == "seca":
                ll_colors.append(VIGNO_ACCENT_GOLD)
            elif hidrica == "normal":
                ll_colors.append(VIGNO_GRAY_DARK)
            else:
                ll_colors.append(VIGNO_GRAY_LIGHT)

    vals_validos_ll = [v for v, lbl in zip(ll_vals, ll_labels, strict=False) if lbl != "Incompleta"]
    promedio_ll = sum(vals_validos_ll) / len(vals_validos_ll) if vals_validos_ll else 0

    bars_ll = ax_ll.bar(ll_temporadas, ll_vals, color=ll_colors, zorder=3)
    ax_ll.axhline(
        promedio_ll,
        color=VIGNO_BLACK,
        linestyle="--",
        zorder=4,
        label=f"Promedio: {promedio_ll:.1f} mm",
    )
    ax_ll.set_title(
        f"Lluvia Invernal - {info['nombre']}", fontweight="bold", color=VIGNO_ACCENT_BLUE, pad=15
    )
    ax_ll.set_ylabel("mm Acumulados")
    ax_ll.set_ylim(0, max(vals_validos_ll) * 1.25 if vals_validos_ll else 800)
    ax_ll.tick_params(axis="x", rotation=45)
    ax_ll.grid(axis="y", linestyle=":", alpha=0.6)
    ax_ll.spines["top"].set_visible(False)
    ax_ll.spines["right"].set_visible(False)
    ax_ll.legend(loc="upper left")

    for bar, lbl in zip(bars_ll, ll_labels, strict=False):
        if lbl == "Incompleta":
            ax_ll.text(
                bar.get_x() + bar.get_width() / 2,
                20,
                "Incompleta",
                ha="center",
                va="bottom",
                rotation=90,
                color=VIGNO_BLACK,
                fontweight="bold",
                fontsize=11,
                fontstyle="italic",
            )
        else:
            yval = bar.get_height()
            ax_ll.text(
                bar.get_x() + bar.get_width() / 2,
                yval + 10,
                f"{yval:.1f}",
                ha="center",
                va="bottom",
                color=VIGNO_BLACK,
                fontweight="bold",
                fontsize=10,
            )
            ax_ll.text(
                bar.get_x() + bar.get_width() / 2,
                20,
                lbl,
                ha="center",
                va="bottom",
                rotation=90,
                color=VIGNO_WHITE if lbl != "Sin clasificar" else VIGNO_BLACK,
                fontweight="bold",
                fontsize=11,
            )

plt.suptitle("Comparación entre Temporadas", fontsize=18, fontweight="bold")
g1_path = output_dir / "E1C53_01_comparacion_temporadas.png"
plt.savefig(g1_path, dpi=300, bbox_inches="tight")
plt.close()

# Gráfico 2: Compara la diferencia de temperatura día a día en el período común de ambas estaciones.
df_list = []
for res in resumenes:
    est_id = res["estacion_id"]
    if est_id not in estaciones_info:
        continue
    for d in res["dias"]:
        df_list.append(
            {
                "estacion": est_id,
                "fecha": d["fecha"],
                "tmax": d["tmax"],
                "tmin": d["tmin"],
                "tmedia": d["tmedia"],
            }
        )

df = pd.DataFrame(df_list)
df["fecha"] = pd.to_datetime(df["fecha"])

df_pivot = df.pivot(index="fecha", columns="estacion", values=["tmax", "tmin", "tmedia"]).dropna()

arenal = "cp-villavicencio-01"
sc = "cp-san-clemente-01"

dif_tmax = df_pivot["tmax"][arenal] - df_pivot["tmax"][sc]
dif_tmin = df_pivot["tmin"][arenal] - df_pivot["tmin"][sc]
dif_tmedia = df_pivot["tmedia"][arenal] - df_pivot["tmedia"][sc]

metricas = {"T. Máxima": dif_tmax, "T. Mínima": dif_tmin, "T. Media": dif_tmedia}

res_comp = []
for nom, dif in metricas.items():
    dias_totales = len(dif)
    dias_mas_calidos = (dif > 0).sum()
    pct_calido = dias_mas_calidos / dias_totales * 100
    dif_media = dif.mean()
    res_comp.append({"Variable": nom, "Dif Media": dif_media, "Pct Cálido": pct_calido})

res_df = pd.DataFrame(res_comp)

fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(14, 6), layout="constrained")

x = np.arange(len(res_df))
width = 0.5

bars1 = ax1.bar(x, res_df["Dif Media"], width, color=VIGNO_TERRACOTTA, zorder=3)
ax1.axhline(0, color=VIGNO_BLACK, linewidth=1)
ax1.set_title(
    "Diferencia Media de Temperatura\n(El Arenal - San Clemente)", fontweight="bold", pad=15
)
ax1.set_ylabel("Diferencia (°C)")
ax1.set_xticks(x)
ax1.set_xticklabels(res_df["Variable"])
ax1.grid(axis="y", linestyle=":", alpha=0.6)
ax1.spines["top"].set_visible(False)
ax1.spines["right"].set_visible(False)

for bar in bars1:
    yval = bar.get_height()
    ax1.text(
        bar.get_x() + bar.get_width() / 2,
        yval + (0.05 if yval > 0 else -0.15),
        f"{yval:+.1f} °C",
        ha="center",
        va="bottom" if yval > 0 else "top",
        fontweight="bold",
        fontsize=11,
    )

bars2 = ax2.bar(x, res_df["Pct Cálido"], width, color=VIGNO_ACCENT_GOLD, zorder=3)
ax2.axhline(50, color=VIGNO_GRAY_DARK, linestyle="--", label="50% (Equidad)")
ax2.set_title(
    "Frecuencia en que El Arenal es más cálido\n(% de días en el período común)",
    fontweight="bold",
    pad=15,
)
ax2.set_ylabel("% de Días")
ax2.set_ylim(0, max(res_df["Pct Cálido"]) * 1.2 if max(res_df["Pct Cálido"]) > 0 else 100)
ax2.set_xticks(x)
ax2.set_xticklabels(res_df["Variable"])
ax2.grid(axis="y", linestyle=":", alpha=0.6)
ax2.spines["top"].set_visible(False)
ax2.spines["right"].set_visible(False)
ax2.legend()

for bar in bars2:
    yval = bar.get_height()
    ax2.text(
        bar.get_x() + bar.get_width() / 2,
        yval + 2,
        f"{yval:.1f} %",
        ha="center",
        fontweight="bold",
        fontsize=11,
    )

fecha_min = df_pivot.index.min().date().strftime("%d/%m/%Y")
fecha_max = df_pivot.index.max().date().strftime("%d/%m/%Y")
plt.suptitle(
    f"Comparación Sistemática de Estaciones\nPeríodo común: {fecha_min} al {fecha_max}",
    fontsize=16,
    fontweight="bold",
)
g2_path = output_dir / "E1C53_02_comparacion_estaciones.png"
plt.savefig(g2_path, dpi=300, bbox_inches="tight")
plt.close()